In [0]:
pip install pandas

In [0]:
import pandas

In [0]:
df = pandas.read_csv("https://raw.githubusercontent.com/Bhevendra/ML-Datasets/refs/heads/main/retail_data/sales_orders.csv")

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
df = spark.createDataFrame(df)


In [0]:
from pyspark.sql.functions import col, from_json, schema_of_json

from pyspark.sql.functions import col, from_json, explode, schema_of_json, lit

# ---------------------------
# SCHEMA INFERENCE FOR ordered_products
# ---------------------------
sample_ordered = df.select("ordered_products") \
    .filter(col("ordered_products").isNotNull()) \
    .first()[0]

ordered_schema = schema_of_json(lit(sample_ordered))

# ---------------------------
# PARSE + EXPLODE
# ---------------------------
df_parsed = df.withColumn(
    "ordered_products_json",
    from_json(col("ordered_products"), ordered_schema)
)

df_exploded = df_parsed.withColumn(
    "product",
    explode(col("ordered_products_json"))
)

# ---------------------------
# SCHEMA INFERENCE FOR promotion_info
# ---------------------------
promo_sample = df_exploded.select("product.promotion_info") \
    .filter(col("product.promotion_info").isNotNull()) \
    .first()[0]

promo_schema = schema_of_json(lit(promo_sample))

df_fixed = df_exploded.withColumn(
    "promo",
    from_json(col("product.promotion_info"), promo_schema)
)

# ---------------------------
# FINAL FLATTEN
# ---------------------------
df_flat = df_fixed.select(
    "customer_id",
    "customer_name",
    "order_number",
    "order_datetime",
    "number_of_line_items",

    col("product.curr").alias("curr"),
    col("product.id").alias("product_id"),
    col("product.name").alias("product_name"),
    col("product.price").cast("int").alias("price"),
    col("product.qty").cast("int").alias("qty"),
    col("product.unit").alias("unit"),

    col("promo.promo_disc"),
    col("promo.promo_id"),
    col("promo.promo_item"),
    col("promo.promo_qty")
)

display(df_flat)

In [0]:
df_flat.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("batch_1.data.sales_orders")

- Fact vs Dimension table
- Normalization vs Denormalization
- Slowly Changing Dimensions (SCD)
- Keys
